In [0]:
# Sample batting data from World Cup 2023 (practice data)
data = [
    ("Virat Kohli", "India", "Semi Final", 117),
    ("Rohit Sharma", "India", "Semi Final", 47),
    ("Travis Head", "Australia", "Final", 137),
    ("Virat Kohli", "India", "Final", 54),
    ("Rohit Sharma", "India", "Final", 47),
    ("Travis Head", "Australia", "Final", 137),   # duplicate row
    ("Glenn Maxwell", "Australia", "League", None) # missing runs
]

df = spark.createDataFrame(data, ["player", "team", "match", "runs"])
display(df)

In [0]:
from pyspark.sql.functions import col

clean = (df
    .dropDuplicates()            # remove the repeated Travis Head row
    .fillna({"runs": 0}))        # replace missing runs with 0

display(clean)

In [0]:
from pyspark.sql.functions import sum as _sum

totals = (clean
    .groupBy("player", "team")
    .agg(_sum("runs").alias("total_runs"))
    .orderBy(col("total_runs").desc()))

display(totals)


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import dense_rank

w = Window.partitionBy("team").orderBy(col("total_runs").desc())

top_per_team = (totals
    .withColumn("rank", dense_rank().over(w))
    .filter(col("rank") == 1))

display(top_per_team)